<a href="https://colab.research.google.com/github/souvikkai/souvik-ai-pm-portfolio/blob/main/day22-qlora-finetuning-lab/Day_22_QLoRA_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch, platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory GB: 15.64


In [2]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.1 MB/s eta 0:00:00


In [3]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    get_peft_model,
)

from trl import SFTTrainer
from datasets import Dataset

print("All imports successful.")

All imports successful.


In [5]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded.
Pad token: <|endoftext|>
EOS token: <|im_end|>


In [6]:
raw_examples = [
    {
        "question": "Why is HBM bandwidth important for AI inference?",
        "answer": "HBM bandwidth is critical because GPUs must continuously move activations and KV cache data during inference. Insufficient bandwidth can starve compute units and increase latency."
    },
    {
        "question": "What is KV cache in transformers?",
        "answer": "KV cache stores attention keys and values from previous tokens so the model does not recompute them during autoregressive generation."
    },
    {
        "question": "Why does P99 latency matter in AI systems?",
        "answer": "P99 latency captures worst-case user experience. High tail latency can make AI applications feel unreliable even if average latency is low."
    },
    {
        "question": "Why use LoRA instead of full fine-tuning?",
        "answer": "LoRA reduces GPU memory and training cost by updating small low-rank adapter matrices instead of all model weights."
    },
]

formatted_examples = []

for ex in raw_examples:
    messages = [
        {
            "role": "system",
            "content": "You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM."
        },
        {
            "role": "user",
            "content": ex["question"]
        },
        {
            "role": "assistant",
            "content": ex["answer"]
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    formatted_examples.append({"text": text})

dataset = Dataset.from_list(formatted_examples)

print(dataset)
print(dataset[0]["text"])

Dataset({
    features: ['text'],
    num_rows: 4
})
<|im_start|>system
You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM.<|im_end|>
<|im_start|>user
Why is HBM bandwidth important for AI inference?<|im_end|>
<|im_start|>assistant
HBM bandwidth is critical because GPUs must continuously move activations and KV cache data during inference. Insufficient bandwidth can starve compute units and increase latency.<|im_end|>



In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(bnb_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [8]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded.")
print("Device:", model.device)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.
Device: cuda:0


In [9]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [10]:
training_args = TrainingArguments(
    output_dir="./qwen-day22-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,

In [11]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Trainer created.")

Adding EOS to train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Trainer created.


In [12]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [13]:
from collections import Counter

dtype_counts = Counter()

for name, param in model.named_parameters():
    if param.requires_grad:
        dtype_counts[str(param.dtype)] += param.numel()

print(dtype_counts)

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype, param.shape)
        break

Counter({'torch.bfloat16': 540672})
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.bfloat16 torch.Size([8, 896])


In [14]:
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

from collections import Counter

dtype_counts = Counter()
for name, param in model.named_parameters():
    if param.requires_grad:
        dtype_counts[str(param.dtype)] += param.numel()

print(dtype_counts)

Counter({'torch.float32': 540672})


In [15]:
trainer.train()

Step,Training Loss
1,5.927675
2,5.927675
3,5.649183
4,5.485950
5,5.369934


TrainOutput(global_step=5, training_loss=5.672083473205566, metrics={'train_runtime': 7.7417, 'train_samples_per_second': 2.583, 'train_steps_per_second': 0.646, 'total_flos': 3574351554048.0, 'train_loss': 5.672083473205566})

In [16]:
prompt = "Why is KV cache important for LLM inference?"

messages = [
    {
        "role": "system",
        "content": "You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.7,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

system
You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM.
user
Why is KV cache important for LLM inference?
assistant
KV Cache (Key-Value Cache) is crucial for Liquid Learning Inference (LLI) for several reasons:

1. High Throughput: KV Cache allows for high throughput, enabling the storage of large amounts of data in memory.

2. Efficient Data Access: It provides fast access to key-value pairs through its highly parallelized architecture.

3. Fault Tolerance: The KV Cache ensures fault tolerance


In [17]:
prompt = "What is KV cache in transformers?"

messages = [
    {
        "role": "system",
        "content": "You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


system
You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM.
user
What is KV cache in transformers?
assistant
KV Cache (Key-Value) Cache is a specialized memory structure used by Transformers, which is a type of neural network architecture designed for processing large amounts of data efficiently. In Transformers, the KV Cache serves as an additional layer of memory that stores key-value pairs during the forward pass.

The KV Cache works in two main ways:

1. **Memory Access**: It allows for efficient access to the state


In [18]:
gpu_stats = torch.cuda.get_device_properties(0)

total_memory = gpu_stats.total_memory / 1024**3
reserved_memory = torch.cuda.max_memory_reserved() / 1024**3
allocated_memory = torch.cuda.max_memory_allocated() / 1024**3

print(f"GPU: {gpu_stats.name}")
print(f"Total GPU memory: {total_memory:.2f} GB")
print(f"Max reserved memory: {reserved_memory:.2f} GB")
print(f"Max allocated memory: {allocated_memory:.2f} GB")

GPU: Tesla T4
Total GPU memory: 14.56 GB
Max reserved memory: 1.53 GB
Max allocated memory: 1.08 GB


In [19]:
model.save_pretrained("./qwen-day22-lora-adapter")
tokenizer.save_pretrained("./qwen-day22-lora-adapter")

print("Saved LoRA adapter and tokenizer.")

Saved LoRA adapter and tokenizer.


In [20]:
!ls -lh ./qwen-day22-lora-adapter

total 13M
-rw-r--r-- 1 root root 1.1K May 19 21:44 adapter_config.json
-rw-r--r-- 1 root root 2.1M May 19 21:44 adapter_model.safetensors
-rw-r--r-- 1 root root 2.5K May 19 21:44 chat_template.jinja
-rw-r--r-- 1 root root 5.1K May 19 21:44 README.md
-rw-r--r-- 1 root root  665 May 19 21:44 tokenizer_config.json
-rw-r--r-- 1 root root  11M May 19 21:44 tokenizer.json


In [21]:
!zip -r qwen-day22-lora-adapter.zip qwen-day22-lora-adapter
!ls -lh qwen-day22-lora-adapter.zip

  adding: qwen-day22-lora-adapter/ (stored 0%)
  adding: qwen-day22-lora-adapter/adapter_config.json (deflated 58%)
  adding: qwen-day22-lora-adapter/chat_template.jinja (deflated 71%)
  adding: qwen-day22-lora-adapter/README.md (deflated 65%)
  adding: qwen-day22-lora-adapter/tokenizer_config.json (deflated 59%)
  adding: qwen-day22-lora-adapter/adapter_model.safetensors (deflated 10%)
  adding: qwen-day22-lora-adapter/tokenizer.json (deflated 81%)
-rw-r--r-- 1 root root 4.0M May 19 21:46 qwen-day22-lora-adapter.zip
